# FLAG 2027 — NB-63w: ResNet293-LM + ECAPA1024-LM of the ~21k auxiliary identities (ENC-SCALE-01 stage A)

**Why.** Track 3 (CB 17.70) gets most of its Bangla margin from **voice-encoder diversity**: sim, sam100ft, ecapa, r293 and their
concatenation `cat4`, each in its own branch. Our CC head uses SimAM p + ft only; ENC-CC-01 screened `cat4` WITHOUT auxiliary heads
(+0.04, a weak screen) because the auxiliary voices lacked the two other encoders. This notebook adds them for every auxiliary voice
row, so ENC-SCALE-01 stage B can train `cat4` (and one-branch-per-encoder) CC heads WITH the auxiliary sources.

| output | rows |
|---|---|
| `feats_aux_voice/<vox / vox2b / vb>/voice_rn293.npy`, `voice_ecapa1024.npy` | same order as the source's `voice_meta.csv` (NaN = not fetched or LID-dropped) |

**Wiring check:** frozen SimAM (sam100p / sam100f) on the re-fetched centre 12 s reproduces the stored arrays (mean cosine >= 0.99).
**Rules:** VoxBlink CC BY-NC-SA: notebook, output and dataset **private**. No Bengali voice is fetched (each source's LID rule).
**Inputs:** `flag2027-frames-index-p`, `flag2027-lab-vb-p`, `flag2027-lab-data-p`, `flag2027-feats-vox2b-p`. **GPU T4**, **Internet ON**,
secret `HF_TOKEN`. **Time:** about 6-8 h (resumable per shard / tar). Upload to `VFEAT_UPLOAD` when set.

In [ ]:
print("[boot] FLAG_63w_aux_voice_feats | python", __import__("sys").version.split()[0], "| start", __import__("time").strftime("%Y-%m-%d %H:%M:%S"), flush=True)  # heartbeat rule: first output, before any import
import os, sys, io, re, json, time, shutil, tarfile, threading, types
from pathlib import Path
import numpy as np, pandas as pd
LOCAL = os.environ.get("FLAG_LOCAL") == "1"
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
WORK = Path(os.environ.get("FLAG_OUT", "/kaggle/working"))
OUT = WORK / "feats_aux_voice"
for d in (OUT, OUT / "vb", OUT / "vox", OUT / "chunks_vb", OUT / "chunks_vox"):
    d.mkdir(parents=True, exist_ok=True)
CACHE = Path(os.environ.get("FLAG_CACHE", "/tmp/flag_cache")); CACHE.mkdir(parents=True, exist_ok=True)
SMOKE = LOCAL or os.environ.get("FLAG_SMOKE") == "1"
SMOKE_N = int(os.environ.get("AUDIO_SMOKE_N", "20"))                 # smoke: videos of the first VoxBlink shard
SMOKE_TARS = int(os.environ.get("AUDIO_SMOKE_TARS", "2"))            # smoke: VoxCeleb2 tars streamed
TIME_LIMIT_H = float(os.environ.get("AUDIO_TIME_LIMIT_H", "10.5"))
THREADS = int(os.environ.get("AUDIO_THREADS", "12"))
TAR_THREADS = int(os.environ.get("AUDIO_TAR_THREADS", "4"))
SR, FPS, CROP_SEC, WIRE_SEC = 16000, 25, 3.0, 12.0                  # FLAG_46's FPS; T3 cut = centre 12 s
WIRE_N = int(os.environ.get("AUDIO_WIRE_N", "48"))
L_CROP, L_WIRE = int(CROP_SEC * SR), int(WIRE_SEC * SR)
T_START = time.time()


def log(*a):
    print(time.strftime("%H:%M:%S"), f"[{(time.time() - T_START) / 3600:4.1f}h]", *a, flush=True)


def time_left():
    return time.time() - T_START < TIME_LIMIT_H * 3600


def centre_crop(w, L):                                               # = FLAG_12's centre_crop
    if len(w) <= L:
        return w
    s = (len(w) - L) // 2
    return w[s:s + L]


def to_i16(w, L):
    """centre L samples as int16 (zero padded) + the valid length."""
    c = centre_crop(w, L)
    out = np.zeros(L, np.int16); out[:len(c)] = np.clip(np.round(c * 32767.0), -32768, 32767).astype(np.int16)
    return out, len(c)


log("aux voice features (rn293, ecapa1024) | smoke", SMOKE, "| threads", THREADS, TAR_THREADS, "| time limit", TIME_LIMIT_H, "h | crop", CROP_SEC, "s")

In [ ]:
# PyAV decodes AAC / mp4 audio; WeSpeaker (commit 9fecd6c, as the T3 notebooks) only for the wiring check (--no-deps: its pins
# would replace Kaggle's torch)
if not LOCAL:
    !pip -q install av kaldiio silero-vad 2>&1 | tail -1
    !pip -q install --no-deps "git+https://github.com/wenet-e2e/wespeaker.git@9fecd6cb4f47475d01761d87c826298dff4ef18c" 2>&1 | tail -1
import requests, av, torch
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HDR = {"Authorization": f"Bearer {os.environ['HF_TOKEN']}"} if os.environ.get("HF_TOKEN") else {}
log("av", av.__version__, "| torch", torch.__version__, "| device", DEV, "| HF token", bool(HDR))
# resume: chunk files of an earlier run (this notebook's output attached)
n_res = 0
for prev in ([] if LOCAL else [p for p in IN.rglob("feats_aux_voice") if p.is_dir() and (p / "chunks_vb").exists()]):
    for f in list((prev / "chunks_vb").glob("*.npz")) + list((prev / "chunks_vox").glob("*.npz")) + list(prev.glob("wire_*.npz")):
        tgt = OUT / f.relative_to(prev)
        if not tgt.exists():
            tgt.parent.mkdir(parents=True, exist_ok=True); shutil.copy(f, tgt); n_res += 1
log("resume: chunk files copied from an earlier run:", n_res)

## 1. Voice rows of each source (LID-dropped speakers excluded)

In [ ]:
# ============================================================================ the voice rows of each source, minus the LID-dropped speakers
def one_dir(fname, parent):
    hits = sorted({p.parent for p in IN.rglob(fname) if p.parent.name == parent})
    assert hits, f"{fname} under a folder named {parent} not found: attach its dataset"
    return hits[0]


dt = {"spk": str, "video": str, "utt": str}
D_VB, D_V37, D_V2B = one_dir("voice_meta.csv", "feats_vb"), one_dir("voice_meta.csv", "feats_vox2"), one_dir("voice_meta.csv", "feats_vox2b")
D_V37T3, D_LID37 = one_dir("voice_sam100p.npy", "feats_vox2t3"), one_dir("lid_summary.json", "feats_vox2lid")
drop = lambda f: set(json.loads(Path(f).read_text())["speakers_dropped_ids"])
META = {"vb": pd.read_csv(D_VB / "voice_meta.csv", dtype=dt), "vox": pd.read_csv(D_V37 / "voice_meta.csv", dtype=dt),
        "vox2b": pd.read_csv(D_V2B / "voice_meta.csv", dtype=dt)}
DROP = {"vb": drop(D_VB / "lid_summary.json"), "vox": drop(D_LID37 / "lid_summary.json"), "vox2b": drop(D_V2B / "lid_summary.json")}
ARRS = {"vb": D_VB, "vox": D_V37T3, "vox2b": D_V2B}                  # stored voice_sam100p / voice_sam100f, rows = META rows
ROW = {s: {k: i for i, k in enumerate(zip(m.spk, m.video, m.utt))} for s, m in META.items()}
WANT = {s: {k for k in ROW[s] if k[0] not in DROP[s]} for s in META}
for s in META:
    log(f"{s}: {len(META[s])} voice rows, {len(DROP[s])} LID-dropped speakers -> {len(WANT[s])} rows, "
        f"{len({k[0] for k in WANT[s]})} speakers | arrays {ARRS[s]}")
IDX_VB = next(p.parent for p in IN.rglob("shard_listing.csv"))
SEL = pd.read_csv(IDX_VB / "selection.csv", dtype=dt)
TSV = pd.read_csv(IDX_VB / "timestamps.csv", dtype=dt)
LSV = pd.read_csv(IDX_VB / "shard_listing.csv")
LSV["spk"], LSV["video"] = LSV.member.str.split("/").str[0], LSV.member.str.split("/").str[1].str[:-4]
JOB = SEL.merge(TSV, on=["spk", "video", "utt"]).merge(LSV[["part", "shard", "spk", "video", "off", "size"]], on=["part", "shard", "spk", "video"])
JOB = JOB[[k in WANT["vb"] for k in zip(JOB.spk, JOB.video, JOB.utt)]].drop_duplicates(["spk", "video", "utt"]).reset_index(drop=True)
log(f"VoxBlink: {len(JOB)}/{len(WANT['vb'])} wanted rows with selection + timestamp + member offset, {JOB.groupby(['part', 'shard']).ngroups} shards")

## 2. The two encoders

In [ ]:
T3CELL_SRC = '# ============================================================================ T3ENC-01 encoders: the encoder family of Track 3\'s VoxBlink system, frozen\n# Track 3 (a teammate\'s VoxBlink pipeline, CB Bangla ng ~16 / g ~25) used these four encoders; here they run on OUR rows\n# (MAV-Celeb v4 train, v1/v2, dev lists; and VoxCeleb2 in FLAG_41), so we can test locally whether the encoders and a\n# cross-modal head carry the gain WITHOUT VoxBlink data (docs/PLAN_VOXBLINK_2026-10-07.md, V1). Every sample on its own.\n#   farl16V   FaRL ViT-B/16 LAION-Face 20M **ep16** (github.com/FacePerceiver/FaRL release, 652 MB) on openai CLIP ViT-B/16\n#             (CLIP preprocessing: 224 bicubic, centre crop); 512-d image embedding (our existing `farl` is the ep64 copy)\n#   clipLV    openai/clip-vit-large-patch14 @ 32bd642 (transformers): get_image_features, 768-d\n#   sam100pA  WeSpeaker voxblink2_samresnet100 (VoxBlink2 pretrained), official ModelScope zip (wenet/wespeaker_pretrained_models)\n#   sam100fA  WeSpeaker voxblink2_samresnet100_ft (VoxBlink2 pretrained -> VoxCeleb2 large-margin fine-tune)\n#             both: wespeaker fbank + CMN, int16-scaled waveform (as our wssim34 / wsr221 cells), 256-d\nimport zipfile as _zf\n\nFARL16_URL = "https://github.com/FacePerceiver/FaRL/releases/download/pretrained_weights/FaRL-Base-Patch16-LAIONFace20M-ep16.pth"\nFARL16_SIZE = 652165542\nCLIPL_ID, CLIPL_REV = "openai/clip-vit-large-patch14", "32bd64288804d66eefd0ccbe215aa642df71cc41"\nWESPK_TREE = "https://modelscope.cn/api/v1/datasets/wenet/wespeaker_pretrained_models/oss/tree"\nSAM_ZIPS = {"sam100pA": ("voxblink2_samresnet100.zip", 186890339), "sam100fA": ("voxblink2_samresnet100_ft.zip", 186890839)}\nT3_STATS = {}\n\n\ndef fetch(url, dest, size=None):\n    dest = Path(dest)\n    if dest.exists() and (size is None or dest.stat().st_size == size):\n        return dest\n    torch.hub.download_url_to_file(url, str(dest) + ".part", progress=False)\n    Path(str(dest) + ".part").replace(dest)\n    assert size is None or dest.stat().st_size == size, f"{dest.name}: {dest.stat().st_size} bytes, expected {size}"\n    return dest\n\n\nclass FaRL16Face:\n    kind = "face"\n\n    def __init__(self):\n        import clip\n        m, self.pre = clip.load("ViT-B/16", device="cpu", download_root=str(CACHE / "clip"))\n        sd = torch.load(fetch(FARL16_URL, CACHE / "FaRL-ep16.pth", FARL16_SIZE), map_location="cpu", weights_only=False)["state_dict"]\n        vis = [k for k in m.state_dict() if k.startswith("visual.")]\n        hit = [k for k in vis if k in sd and sd[k].shape == m.state_dict()[k].shape]\n        assert len(hit) == len(vis), f"FaRL ep16 covers {len(hit)}/{len(vis)} visual tensors"\n        m.load_state_dict(sd, strict=False)\n        self.m = m.visual.float().to(DEV).eval()\n\n    def prep(self, im):\n        return self.pre(im)\n\n    @torch.no_grad()\n    def __call__(self, x, use_amp=True):\n        return {"": self.m(x.to(DEV)).float().cpu().numpy()}\n\n\nclass CLIPLFace:\n    kind = "face"\n\n    def __init__(self):\n        from transformers import CLIPModel, CLIPImageProcessor\n        self.proc = CLIPImageProcessor.from_pretrained(CLIPL_ID, revision=CLIPL_REV)\n        self.m = CLIPModel.from_pretrained(CLIPL_ID, revision=CLIPL_REV, torch_dtype=torch.float32).to(DEV).eval()\n\n    def prep(self, im):\n        return self.proc(images=im.convert("RGB"), return_tensors="pt").pixel_values[0]\n\n    @torch.no_grad()\n    def __call__(self, x, use_amp=True):\n        f = self.m.get_image_features(pixel_values=x.to(DEV))\n        f = f.pooler_output if hasattr(f, "pooler_output") else f          # transformers 5 may return an output object\n        return {"": f.float().cpu().numpy()}\n\n\ndef wespeaker_dir(zip_name, size):\n    """The official zip (ModelScope, as wespeaker\'s own hub does), unpacked without the __MACOSX entries."""\n    d = CACHE / "wespeaker" / zip_name[:-4]\n    if (d / "avg_model.pt").exists() and (d / "config.yaml").exists():\n        return d\n    d.mkdir(parents=True, exist_ok=True)\n    import requests, shutil as _sh\n    hit = [p for p in Path("/kaggle/input").rglob(zip_name) if p.stat().st_size == size] if Path("/kaggle/input").exists() else []\n    if hit:                                                            # a weights dataset is attached: no network\n        z = d / zip_name\n        if not (z.exists() and z.stat().st_size == size):\n            _sh.copy(hit[0], z)\n    else:\n        for _a in range(6):                                            # modelscope.cn times out from Kaggle now and then\n            try:\n                tree = requests.get(WESPK_TREE, timeout=120).json()["Data"]\n                url = [it["Url"] for it in tree if it.get("Key") == zip_name][0]\n                z = fetch(url, d / zip_name, size)\n                break\n            except Exception as _e:\n                if _a == 5:\n                    raise\n                print(f"wespeaker {zip_name}: download failed ({_e!r}); retry {_a + 1}/5 in {30 * (_a + 1)} s", flush=True)\n                time.sleep(30 * (_a + 1))\n    with _zf.ZipFile(z) as zf:\n        for mem in zf.namelist():\n            if not mem.startswith("__MACOSX") and not mem.endswith("/"):\n                (d / Path(mem).name).write_bytes(zf.read(mem))\n    # some releases (voxceleb_resnet293_LM) name the pair <model>.pt / <model>.yaml instead of avg_model.pt / config.yaml\n    pts, ymls = sorted(d.glob("*.pt")), sorted(d.glob("*.yaml"))\n    if not (d / "avg_model.pt").exists() and len(pts) == 1:\n        pts[0].rename(d / "avg_model.pt")\n    if not (d / "config.yaml").exists() and len(ymls) == 1:\n        ymls[0].rename(d / "config.yaml")\n    assert (d / "avg_model.pt").exists() and (d / "config.yaml").exists(), f"{zip_name}: unexpected content {zf.namelist()[:6]}"\n    return d\n\n\nclass SimAMVoice:\n    kind = "voice"\n\n    def __init__(self, name):\n        for mod, attr in (("wespeaker.frontend.s3prl", "S3prlFrontend"), ("wespeaker.frontend.whisper_encoder", "whisper_encoder"),\n                          ("wespeaker.frontend.w2vbert", "W2VBertFrontend"), ("wespeaker.diar.umap_clusterer", "cluster"),\n                          ("wespeaker.diar.extract_emb", "subsegment")):\n            if mod not in sys.modules:                     # SSL frontends / diarisation helpers: not used by fbank models\n                stub = types.ModuleType(mod); setattr(stub, attr, None); sys.modules[mod] = stub\n        import wespeaker\n        self.m = wespeaker.load_model(str(wespeaker_dir(*SAM_ZIPS[name])))\n        self.m.set_device("cuda:0" if DEV.type == "cuda" else "cpu")\n        T3_STATS[name] = {"params_M": round(sum(p.numel() for p in self.m.model.parameters()) / 1e6, 1)}\n\n    @torch.no_grad()\n    def __call__(self, w, use_amp=True):\n        pcm = torch.from_numpy(np.ascontiguousarray(w, dtype=np.float32) * 32768.0)[None]\n        feats = self.m.compute_features(pcm, sample_rate=16000, cmn=True)\n        out = self.m.model(feats.to(self.m.device))\n        out = out[-1] if isinstance(out, tuple) else out\n        return {"": out[0].reshape(-1).float().cpu().numpy()}\n\n\nNEW = {"farl16V": FaRL16Face, "clipLV": CLIPLFace, "sam100pA": lambda: SimAMVoice("sam100pA"),\n       "sam100fA": lambda: SimAMVoice("sam100fA")}\n\n\ndef build(name):\n    return NEW[name]()\n\n\ndef release(keep_ib=False):\n    import gc\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\nARRAYS_NEW = {"farl16V": ("face", {"": "farl16"}), "clipLV": ("face", {"": "clipL"}),\n              "sam100pA": ("voice", {"": "sam100p"}), "sam100fA": ("voice", {"": "sam100f"})}\nMAIN_NEW = {k: "" for k in NEW}\nRESHAPE = {}\n'

# ============================================================================ the two encoders (T3 notebooks' loader; official ModelScope zips)
exec(compile(T3CELL_SRC, "t3enc_cell", "exec"))                       # SimAMVoice, wespeaker_dir, fetch, SAM_ZIPS
SAM_ZIPS.update({"rn293A": ("voxceleb_resnet293_LM.zip", 124116126), "ecapa1024A": ("voxceleb_ECAPA1024_LM.zip", 67665905)})
VENC = {"rn293": SimAMVoice("rn293A"), "ecapa1024": SimAMVoice("ecapa1024A")}
DIMS_V = {"rn293": 256, "ecapa1024": 192}


def save_emb(path, items):
    """items = [((spk, video, utt), float wav)] -> npz: keys + both embeddings of the centre 12 s (the T3 cut)."""
    keys = [k for k, _ in items]
    E = {n: np.zeros((len(items), d), np.float32) for n, d in DIMS_V.items()}
    for i, (_, w) in enumerate(items):
        c = centre_crop(w, L_WIRE)
        for n, enc in VENC.items():
            E[n][i] = enc(c)[""]
    k = np.array(keys, str).reshape(-1, 3)
    tmp = Path(str(path) + ".part.npz")
    np.savez(tmp, spk=k[:, 0], video=k[:, 1], utt=k[:, 2], **E)
    tmp.replace(path)

## 3. VoxBlink: re-fetch (FLAG_63's shard index), FLAG_46's utterance cut, embed the centre 12 s

In [ ]:
# ============================================================================ VoxBlink: FLAG_46's utterance cut of the re-fetched video, centre 3 s
import concurrent.futures as cf
SHARD = "https://huggingface.co/datasets/adollahamini1998/VoxBlink-Video-clean-8-{}/resolve/main/{}"
_TL = threading.local()


def cdn_url(part, shard):
    with requests.get(SHARD.format(part, shard), headers={"Range": "bytes=0-0", **HDR}, allow_redirects=True, timeout=120, stream=True) as r:
        return r.url


def decode_audio(b):
    """The whole audio track as mono float32 at SR (= vb_media.decode_video's audio branch)."""
    out = []
    with av.open(io.BytesIO(b)) as c:
        if not c.streams.audio:
            return np.zeros(0, np.float32)
        res = av.AudioResampler(format="flt", layout="mono", rate=SR)
        for fr in c.decode(c.streams.audio[0]):
            out += [o.to_ndarray().reshape(-1) for o in res.resample(fr)]
        out += [o.to_ndarray().reshape(-1) for o in res.resample(None)]
    return np.concatenate(out).astype(np.float32) if out else np.zeros(0, np.float32)


def vb_job(job):
    (part, shard, spk, video), g = job
    if not hasattr(_TL, "s"):
        _TL.s, _TL.urls = requests.Session(), {}
    for attempt in range(4):
        try:
            url = _TL.urls.get((part, shard)) or _TL.urls.setdefault((part, shard), cdn_url(part, shard))
            off, size = int(g.off.iloc[0]), int(g["size"].iloc[0])
            b = _TL.s.get(url, headers={"Range": f"bytes={off}-{off + size - 1}"}, timeout=300).content
            if len(b) != size:
                _TL.urls.pop((part, shard), None); raise IOError(f"short read {len(b)} / {size}")
            audio = decode_audio(b)
            out = []
            for r in g.itertuples():
                wav = audio[int(r.start / FPS * SR):int((r.end + 1) / FPS * SR)]
                if len(wav) >= SR // 2:                                # FLAG_46 kept utterances of >= 0.5 s
                    out.append(((spk, video, r.utt), wav))
            return out, None
        except Exception as e:
            err = f"{spk}/{video}: {type(e).__name__}: {e}"; time.sleep(3 * (attempt + 1))
    return [], err


def save_chunk(path, items):
    """items = [((spk, video, utt), float wav)] -> npz with the centre CROP_SEC as int16."""
    if not items:
        np.savez(path, spk=np.array([], str), video=np.array([], str), utt=np.array([], str), dur=np.zeros(0, np.float32),
                 n=np.zeros(0, np.int32), wav=np.zeros((0, L_CROP), np.int16)); return
    W, N = zip(*[to_i16(w, L_CROP) for _, w in items])
    k = np.array([x for x, _ in items], str)
    np.savez(path, spk=k[:, 0], video=k[:, 1], utt=k[:, 2], dur=np.array([len(w) / SR for _, w in items], np.float32),
             n=np.array(N, np.int32), wav=np.stack(W))


WIRE = {s: [] for s in META}                                         # (key, centre 12 s) for the wiring check, first WIRE_N per source
order = JOB.groupby(["part", "shard"]).size().sort_index()
if SMOKE:
    order = order.iloc[:1]
STATS_VB = dict(shards=0, videos=0, failed=0, rows=0)
for (part, shard), n in order.items():
    zf = OUT / "chunks_vb" / f"{part}_{shard}.npz"
    if zf.exists():
        continue
    if not time_left():
        log(f"time limit reached: stopping before VoxBlink shard {shard} (attach the output and run again)"); break
    t0 = time.time()
    G = JOB[(JOB.part == part) & (JOB.shard == shard)]
    jobs = list(G.groupby(["part", "shard", "spk", "video"]))
    if SMOKE:
        jobs = jobs[:SMOKE_N]
    got, errs = [], []
    with cf.ThreadPoolExecutor(THREADS) as ex:
        for res, err in ex.map(vb_job, jobs):
            got += res
            if err:
                errs.append(err)
    for k, w in got:
        if len(WIRE["vb"]) < WIRE_N:
            WIRE["vb"].append((k, centre_crop(w, L_WIRE)))
    save_emb(zf, got)
    STATS_VB["shards"] += 1; STATS_VB["videos"] += len(jobs); STATS_VB["failed"] += len(errs); STATS_VB["rows"] += len(got)
    log(f"VoxBlink {shard}: {len(jobs)} videos ({len(errs)} failed), {len(got)}/{len(G)} rows [{time.time() - t0:.0f}s]"
        + (f" | e.g. {errs[0][:120]}" if errs else ""))
log("VoxBlink done this session:", STATS_VB)

## 4. VoxCeleb2: stream the WebDataset tars, embed the centre 12 s

In [ ]:
# ============================================================================ VoxCeleb2 (FLAG_37 + FLAG_53 rows): the same WebDataset tars, centre 3 s
import concurrent.futures as cf
VOICE_URL = "https://huggingface.co/datasets/gaunernst/voxceleb2-dev-wds/resolve/main/voxceleb2-dev-{:04d}.tar"
SRC_OF = {k: "vox" for k in WANT["vox"]}
SRC_OF.update({k: "vox2b" for k in WANT["vox2b"] if k not in SRC_OF})   # a clip in both lists is cached once (as vox)


def decode_m4a(b):
    out = []
    with av.open(io.BytesIO(b)) as c:
        res = av.AudioResampler(format="flt", layout="mono", rate=SR)
        for fr in c.decode(c.streams.audio[0]):
            out += [o.to_ndarray().reshape(-1) for o in res.resample(fr)]
        out += [o.to_ndarray().reshape(-1) for o in res.resample(None)]
    return np.concatenate(out).astype(np.float32) if out else np.zeros(0, np.float32)


def tar_job(t):
    """Stream tar t once (decoding inside the stream, as FLAG_37); keep the wanted clips. -> (t, [(key, src, wav)], error)"""
    err = None
    for attempt in range(4):
        got = []
        try:
            with requests.get(VOICE_URL.format(t), headers=HDR, timeout=300, stream=True) as r:
                r.raise_for_status(); r.raw.decode_content = True
                tf = tarfile.open(fileobj=r.raw, mode="r|")
                for m in tf:
                    if not m.name.endswith(".m4a"):
                        continue
                    k = tuple(m.name[:-4].split("/")[-3:])
                    if k in SRC_OF:
                        got.append((k, SRC_OF[k], decode_m4a(tf.extractfile(m).read())))
            return t, got, None
        except Exception as e:
            err = f"tar {t}: {type(e).__name__}: {e}"; time.sleep(10 * (attempt + 1))
    return t, [], err


found = set()
for f in sorted((OUT / "chunks_vox").glob("tar*.npz")):                    # resume: clips of the tars already done
    Zp = np.load(f)
    found |= set(zip(Zp["spk"], Zp["video"], Zp["utt"]))
tars = [t for t in range(779) if not (OUT / "chunks_vox" / f"tar{t:04d}.npz").exists()]
if SMOKE:
    tars = tars[:SMOKE_TARS]
STATS_VOX = dict(tars=0, failed=0, rows=0)
t0 = time.time()
with cf.ThreadPoolExecutor(TAR_THREADS) as ex:
    it = iter(tars)
    pending = {}
    for t in it:
        pending[ex.submit(tar_job, t)] = t
        if len(pending) >= TAR_THREADS:
            break
    while pending:
        done, _ = cf.wait(pending, return_when=cf.FIRST_COMPLETED)
        for fut in done:
            pending.pop(fut)
            t, got, err = fut.result()
            if err:
                STATS_VOX["failed"] += 1; log("!!", err, "-> skipped (run again to retry)"); continue
            for k, src, w in got:
                if len(WIRE[src]) < WIRE_N and len(w) >= SR // 2:
                    WIRE[src].append((k, centre_crop(w, L_WIRE)))
            items = [(k, w) for k, src, w in got]
            save_emb(OUT / "chunks_vox" / f"tar{t:04d}.npz", items)
            found |= {k for k, _ in items}
            STATS_VOX["tars"] += 1; STATS_VOX["rows"] += len(items)
            if STATS_VOX["tars"] % 20 == 0 or SMOKE:
                log(f"VoxCeleb2: {STATS_VOX['tars']} tars this session, {len(found)}/{len(SRC_OF)} wanted clips cached [{time.time() - t0:.0f}s]")
        if len(found) >= len(SRC_OF):
            log("every wanted VoxCeleb2 clip is cached"); break
        if not time_left():
            log("time limit reached: no new tar (attach the output and run again)"); continue
        for t in it:
            pending[ex.submit(tar_job, t)] = t
            break
log("VoxCeleb2 done this session:", STATS_VOX, "| cached", len(found), "/", len(SRC_OF))

## 5. Wiring check (frozen SimAM vs the stored arrays)

In [ ]:
# ============================================================================ wiring check: frozen SimAM on the re-fetched audio vs the stored arrays
for s in WIRE:                                                       # keep the samples (a resumed run reuses them)
    f = OUT / f"wire_{s}.npz"
    if WIRE[s] and not f.exists():
        k = np.array([x for x, _ in WIRE[s]], str)
        W, N = zip(*[to_i16(w, L_WIRE) for _, w in WIRE[s]])
        np.savez(f, spk=k[:, 0], video=k[:, 1], utt=k[:, 2], n=np.array(N, np.int32), wav=np.stack(W))
exec(compile(T3CELL_SRC, "t3enc_cell", "exec"))                       # SimAMVoice, wespeaker_dir, fetch (T3 notebooks' code)
WIRING = {}
encs = {n: SimAMVoice(n) for n in ("sam100pA", "sam100fA")}
for s in ("vb", "vox", "vox2b"):
    f = OUT / f"wire_{s}.npz"
    if not f.exists():
        WIRING[s] = "no sample"; continue
    Z = np.load(f)
    rows = [ROW[s][k] for k in zip(Z["spk"], Z["video"], Z["utt"])]
    res = {}
    for n, arr in (("sam100pA", "voice_sam100p.npy"), ("sam100fA", "voice_sam100f.npy")):
        ref = np.load(ARRS[s] / arr, mmap_mode="r")[rows].astype(np.float32)
        new = np.stack([encs[n](Z["wav"][i, :Z["n"][i]].astype(np.float32) / 32767.0)[""] for i in range(len(rows))])
        cos = (new * ref).sum(1) / (np.linalg.norm(new, axis=1) * np.linalg.norm(ref, axis=1) + 1e-8)
        res[n] = dict(rows=len(rows), mean_cos=round(float(cos.mean()), 5), min_cos=round(float(cos.min()), 5))
    WIRING[s] = res
    log(f"wiring {s}: {res}")
ok = all(all(r["mean_cos"] >= 0.99 for r in v.values()) for v in WIRING.values() if isinstance(v, dict))
log("WIRING", "OK" if ok else "!! FAILED: the cache does not hold the clips the stored embeddings came from", json.dumps(WIRING))

## 6. One array per source and encoder

In [ ]:
# ============================================================================ chunks -> one array per source and encoder, in voice_meta.csv row order
ASM = {}
for s in META:
    for n, d in DIMS_V.items():
        ASM[(s, n)] = np.full((len(META[s]), d), np.nan, np.float32)
for sub in ("chunks_vb", "chunks_vox"):
    for f in sorted((OUT / sub).glob("*.npz")):
        Z = np.load(f)
        for i, k in enumerate(zip(Z["spk"], Z["video"], Z["utt"])):
            for s in (("vb",) if sub == "chunks_vb" else ("vox", "vox2b")):     # a VoxCeleb2 clip may be in both lists
                r = ROW[s].get(k)
                if r is not None:
                    for n in DIMS_V:
                        ASM[(s, n)][r] = Z[n][i]
done = {}
for (s, n), A in ASM.items():
    (OUT / s).mkdir(exist_ok=True)
    np.save(OUT / s / f"voice_{n}.npy", A.astype(np.float16))
    done[f"{s}/{n}"] = dict(rows=len(A), finite=int(np.isfinite(A).all(1).sum()), wanted=len(WANT[s]))
log("assembled:", json.dumps(done))

## 7. Manifest + upload

In [ ]:
# ============================================================================ manifest + (optional) upload to a private dataset of this account
cover = {s: done[f"{s}/rn293"]["finite"] / max(1, done[f"{s}/rn293"]["wanted"]) for s in META}
man = dict(created=time.strftime("%Y-%m-%d %H:%M"), encoders={"rn293": "voxceleb_resnet293_LM", "ecapa1024": "voxceleb_ECAPA1024_LM"},
           cut="centre 12 s", arrays=done, coverage=cover, wiring=WIRING, smoke=SMOKE, complete=bool(min(cover.values()) >= 0.97))
(OUT / "vfeat_manifest.json").write_text(json.dumps(man, indent=1, default=str))
log("manifest:", json.dumps(man, default=str))
UP = os.environ.get("VFEAT_UPLOAD", "")
if UP and not SMOKE and man["complete"] and ok:
    try:
        import kagglehub
        stage = Path("/tmp/vfeat_upload"); shutil.rmtree(stage, ignore_errors=True); stage.mkdir(parents=True)
        for s in META:
            (stage / s).mkdir()
            for n in DIMS_V:
                os.symlink(OUT / s / f"voice_{n}.npy", stage / s / f"voice_{n}.npy")
        os.symlink(OUT / "vfeat_manifest.json", stage / "vfeat_manifest.json")
        kagglehub.dataset_upload(UP, str(stage), version_notes="ENC-SCALE-01 stage A: rn293 + ecapa1024 of the auxiliary voices")
        log("uploaded ->", UP)
    except Exception as e:
        log(f"upload to {UP} failed ({e!r}); the arrays are in the notebook output")
else:
    log(f"no upload (VFEAT_UPLOAD={UP!r}, smoke={SMOKE}, complete={man['complete']}, wiring ok={ok}): attach the output and run again if incomplete")